In [ ]:

# ============================================================
# CELL 1 — Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")

Mounted at /content/drive
✅ Drive mounted


In [ ]:

# CELL 2 — Imports & Config
# ============================================================
import os, pickle, gc, math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

BASE      = '/content/drive/MyDrive/new obada/'
PKL_DIR   = BASE + 'data/preprocessed/'
MODEL_DIR = BASE + 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)

WINDOW_SIZE = 599
BATCH_SIZE  = 256
EPOCHS      = 30
LR          = 1e-3
WARMUP      = 3
RANDOM_SEED = 42
VAL_SAMPLE  = 80_000   # 40K ON + 40K OFF — fixed indices every run

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

✅ Device: cuda


In [ ]:

# CELL 3 — Load Data
# ============================================================
print("Loading kettle PKLs...")
with open(PKL_DIR + 'kettle_train.pkl', 'rb') as f:
    tr = pickle.load(f)
with open(PKL_DIR + 'kettle_val.pkl', 'rb') as f:
    vl = pickle.load(f)

X_train = tr['X'].astype(np.float32)        # (40554, 599)
y_train = tr['y_label'].astype(np.float32)  # (40554,) — 50/50

X_val   = vl['X'].astype(np.float32)        # (623419, 599)
y_val   = vl['y_label'].astype(np.float32)  # (623419,) — 0.3% ON

print(f"Train : {X_train.shape} | ON={y_train.mean():.3f}")
print(f"Val   : {X_val.shape}   | ON={y_val.mean():.4f}  ({int(y_val.sum())} ON windows)")

# ── Fixed stratified val sample (same every run) ─────────────
rng     = np.random.default_rng(RANDOM_SEED)
idx_on  = np.where(y_val == 1)[0]
idx_off = np.where(y_val == 0)[0]
n_each  = min(VAL_SAMPLE // 2, len(idx_on))   # limited by #ON windows (2126)
val_idx = np.concatenate([
    rng.choice(idx_on,  n_each, replace=False),
    rng.choice(idx_off, n_each, replace=False)
])
val_idx = np.sort(val_idx)

X_val_s = X_val[val_idx]
y_val_s = y_val[val_idx]
print(f"Val sample: {X_val_s.shape} | ON={y_val_s.mean():.3f}  (fixed seed={RANDOM_SEED})")

# ============================================================

Loading kettle PKLs...
Train : (40554, 599) | ON=0.500
Val   : (623419, 599)   | ON=0.0034  (2126 ON windows)
Val sample: (4252, 599) | ON=0.500  (fixed seed=42)


In [ ]:
# ============================================================
# CELL 4 — Dataset & Model
# ============================================================

class NILMDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X).unsqueeze(1)  # (N, 1, 599)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class KettleCNNBiLSTM(nn.Module):
    """
    CNN  → extracts local power-spike patterns
    BiLSTM → captures temporal context (both directions)
    Seq2Point → predicts centre-point label
    """

    def __init__(self,
                 lstm_hidden=128,
                 lstm_layers=2,
                 dropout=0.3):

        super().__init__()

        # ====================================================
        # CNN ENCODER
        # ====================================================

        self.cnn = nn.Sequential(

            nn.Conv1d(
                in_channels=1,
                out_channels=32,
                kernel_size=7,
                padding=3
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Dropout(dropout)
        )

        # ====================================================
        # BiLSTM
        # ====================================================

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # ====================================================
        # CLASSIFICATION HEAD
        # IMPORTANT:
        # NO SIGMOID HERE
        # BCEWithLogitsLoss handles sigmoid internally
        # ====================================================

        self.head = nn.Sequential(

            nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        # x: (B, 1, 599)

        feat = self.cnn(x)                  # (B, 128, 599)

        feat = feat.permute(0, 2, 1)        # (B, 599, 128)

        out, _ = self.lstm(feat)            # (B, 599, 256)

        mid = out[:, out.size(1) // 2, :]  # centre-point

        logits = self.head(mid).squeeze(1)

        return logits


model = KettleCNNBiLSTM().to(device)

print(f"✅ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

✅ Model parameters: 736,385


In [ ]:
# ============================================================
# CELL 5 — Training
# ============================================================
train_loader = DataLoader(
    NILMDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    NILMDataset(X_val_s, y_val_s),
    batch_size=1024,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# Warmup + Cosine Annealing
def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP

    progress = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ✅ FIXED LOSS
criterion = nn.BCEWithLogitsLoss()

# ✅ FIXED AMP API
scaler = torch.amp.GradScaler(
    'cuda',
    enabled=(device.type == 'cuda')
)

best_f1_on = 0.0
best_path  = MODEL_DIR + 'kettle_cnn_bilstm_best.pt'

print("=" * 75)
print(f"{'Ep':>3} {'Loss':>7} {'F1_ON':>7} {'F1_mac':>7} "
      f"{'Prec':>7} {'Rec':>7} {'Thr':>5} {'LR':>9}")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):

    # ========================================================
    # TRAIN
    # ========================================================
    model.train()
    train_loss = 0.0

    for X_b, y_b in train_loader:

        X_b = X_b.to(device, non_blocking=True)
        y_b = y_b.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ✅ FIXED AUTOCAST API
        with torch.amp.autocast(
            'cuda',
            enabled=(device.type == 'cuda')
        ):

            # logits (NO sigmoid inside model)
            logits = model(X_b)

            # BCEWithLogitsLoss expects raw logits
            loss = criterion(logits, y_b)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    scheduler.step()

    # ========================================================
    # VALIDATION
    # ========================================================
    model.eval()

    all_probs  = []
    all_labels = []

    with torch.no_grad():

        for X_b, y_b in val_loader:

            X_b = X_b.to(device, non_blocking=True)

            # logits
            logits = model(X_b)

            # ✅ convert logits → probabilities
            probs = torch.sigmoid(logits).cpu().numpy()

            all_probs.extend(probs)
            all_labels.extend(y_b.numpy().astype(int))

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    # ========================================================
    # THRESHOLD SEARCH
    # ========================================================
    best_t     = 0.5
    best_f1_ep = 0.0

    for t in np.arange(0.2, 0.85, 0.05):

        preds = (all_probs >= t).astype(int)

        f1_on = f1_score(
            all_labels,
            preds,
            pos_label=1,
            average='binary',
            zero_division=0
        )

        if f1_on > best_f1_ep:
            best_f1_ep = f1_on
            best_t     = t

    final_preds = (all_probs >= best_t).astype(int)

    f1_mac = f1_score(
        all_labels,
        final_preds,
        average='macro',
        zero_division=0
    )

    prec = precision_score(
        all_labels,
        final_preds,
        zero_division=0
    )

    rec = recall_score(
        all_labels,
        final_preds,
        zero_division=0
    )

    avg_loss = train_loss / len(train_loader)
    cur_lr   = optimizer.param_groups[0]['lr']

    is_best = best_f1_ep > best_f1_on

    print(
        f"{epoch:3d} "
        f"{avg_loss:7.4f} "
        f"{best_f1_ep:7.4f} "
        f"{f1_mac:7.4f} "
        f"{prec:7.4f} "
        f"{rec:7.4f} "
        f"{best_t:5.2f} "
        f"{cur_lr:9.6f}"
        + ("  ✅ BEST" if is_best else "")
    )

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================
    if is_best:

        best_f1_on = best_f1_ep

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1_on': best_f1_on,
            'f1_macro': f1_mac,
            'threshold': best_t,
            'architecture': 'CNN-BiLSTM Seq2Point',
            'normalization': 'min-max per chunk (no global scaler)',
        }, best_path)

print("=" * 75)
print(f"\n🏆 Best ON-class F1 = {best_f1_on:.4f}")
print(f"💾 Model saved → {best_path}")

 Ep    Loss   F1_ON  F1_mac    Prec     Rec   Thr        LR
  1  0.2109  0.9865  0.9864  0.9756  0.9976  0.80  0.000667  ✅ BEST
  2  0.1260  0.9883  0.9882  0.9801  0.9967  0.70  0.001000  ✅ BEST
  3  0.1196  0.9862  0.9861  0.9814  0.9911  0.80  0.001000
  4  0.1088  0.9850  0.9849  0.9795  0.9906  0.80  0.000997
  5  0.0999  0.9858  0.9857  0.9747  0.9972  0.80  0.000987
  6  0.0936  0.9833  0.9833  0.9808  0.9859  0.75  0.000970
  7  0.0901  0.9830  0.9828  0.9755  0.9906  0.80  0.000947
  8  0.0860  0.9842  0.9840  0.9751  0.9934  0.80  0.000918
  9  0.0832  0.9820  0.9819  0.9758  0.9882  0.75  0.000883
 10  0.0818  0.9853  0.9852  0.9773  0.9934  0.80  0.000843
 11  0.0750  0.9805  0.9802  0.9696  0.9915  0.65  0.000799
 12  0.0721  0.9818  0.9817  0.9719  0.9920  0.75  0.000750
 13  0.0700  0.9789  0.9786  0.9631  0.9953  0.45  0.000698
 14  0.0655  0.9798  0.9795  0.9692  0.9906  0.50  0.000643
 15  0.0619  0.9823  0.9821  0.9724  0.9925  0.55  0.000587
 16  0.0607  0.9823  0.9

In [ ]:
# ============================================================
# FULL VALIDATION ON ENTIRE HOUSE 20
# ============================================================

full_val_loader = DataLoader(
    NILMDataset(X_val, y_val),
    batch_size=1024,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ============================================================
# LOAD BEST MODEL
# ============================================================

ckpt = torch.load(
    best_path,
    map_location=device,
    weights_only=False   # <-- PyTorch 2.6 fix
)

model.load_state_dict(ckpt['model_state_dict'])
model.eval()

THRESHOLD = ckpt['threshold']

# ============================================================
# FULL VALIDATION
# ============================================================

all_probs  = []
all_labels = []

with torch.no_grad():

    for X_b, y_b in full_val_loader:

        X_b = X_b.to(device, non_blocking=True)

        logits = model(X_b)

        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(y_b.numpy().astype(int))

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

preds = (all_probs >= THRESHOLD).astype(int)

# ============================================================
# METRICS
# ============================================================

f1_on = f1_score(
    all_labels,
    preds,
    average='binary',
    pos_label=1,
    zero_division=0
)

f1_mac = f1_score(
    all_labels,
    preds,
    average='macro',
    zero_division=0
)

prec = precision_score(
    all_labels,
    preds,
    zero_division=0
)

rec = recall_score(
    all_labels,
    preds,
    zero_division=0
)

# ============================================================
# RESULTS
# ============================================================

print("=" * 55)
print("FULL HOUSE-20 VALIDATION RESULTS")
print("=" * 55)
print(f"Threshold : {THRESHOLD:.2f}")
print(f"F1 ON     : {f1_on:.4f}")
print(f"F1 Macro  : {f1_mac:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"Total ON  : {all_labels.sum():,}")
print(f"Pred ON   : {preds.sum():,}")
print("=" * 55)

FULL HOUSE-20 VALIDATION RESULTS
Threshold : 0.70
F1 ON     : 0.2748
F1 Macro  : 0.6328
Precision : 0.1593
Recall    : 0.9967
Total ON  : 2,126
Pred ON   : 13,298


In [ ]:
# ============================================================
# THRESHOLD SEARCH ON FULL VALIDATION
# ============================================================

results = []

print("=" * 60)
print(f"{'Thr':>6} {'F1_ON':>10} {'Prec':>10} {'Recall':>10}")
print("=" * 60)

for t in np.arange(0.70, 0.999, 0.01):

    preds = (all_probs >= t).astype(int)

    f1_on = f1_score(
        all_labels,
        preds,
        average='binary',
        pos_label=1,
        zero_division=0
    )

    prec = precision_score(
        all_labels,
        preds,
        zero_division=0
    )

    rec = recall_score(
        all_labels,
        preds,
        zero_division=0
    )

    results.append((t, f1_on, prec, rec))

    print(f"{t:6.2f} {f1_on:10.4f} {prec:10.4f} {rec:10.4f}")

best = max(results, key=lambda x: x[1])

print("\n" + "=" * 60)
print("BEST THRESHOLD ON FULL VALIDATION")
print("=" * 60)
print(f"Threshold : {best[0]:.2f}")
print(f"F1_ON     : {best[1]:.4f}")
print(f"Precision : {best[2]:.4f}")
print(f"Recall    : {best[3]:.4f}")
print("=" * 60)

   Thr      F1_ON       Prec     Recall
  0.70     0.2748     0.1593     0.9967
  0.71     0.2804     0.1632     0.9967
  0.72     0.2859     0.1669     0.9958
  0.73     0.2916     0.1708     0.9958
  0.74     0.2967     0.1743     0.9948
  0.75     0.3017     0.1779     0.9939
  0.76     0.3073     0.1818     0.9939
  0.77     0.3139     0.1864     0.9939
  0.78     0.3200     0.1907     0.9939
  0.79     0.3264     0.1953     0.9934
  0.80     0.3333     0.2002     0.9934
  0.81     0.3399     0.2050     0.9934
  0.82     0.3469     0.2102     0.9934
  0.83     0.3541     0.2155     0.9925
  0.84     0.3615     0.2210     0.9925
  0.85     0.3696     0.2271     0.9925
  0.86     0.3775     0.2331     0.9920
  0.87     0.3866     0.2401     0.9911
  0.88     0.3949     0.2466     0.9911
  0.89     0.4039     0.2537     0.9911
  0.90     0.4147     0.2623     0.9906
  0.91     0.4254     0.2709     0.9901
  0.92     0.4368     0.2804     0.9878
  0.93     0.4499     0.2914     0.9868


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, pickle, math, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

BASE      = '/content/drive/MyDrive/new obada/'
PKL_DIR   = BASE + 'data/preprocessed/'
MODEL_DIR = BASE + 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

print("\nLoading PKLs...")
with open(PKL_DIR + 'kettle_train_w99.pkl', 'rb') as f:
    tr = pickle.load(f)
with open(PKL_DIR + 'kettle_val_w99.pkl', 'rb') as f:
    vl = pickle.load(f)

X_train = tr['X'].astype(np.float32)
y_train = tr['y_label'].astype(np.float32)
X_val   = vl['X'].astype(np.float32)
y_val   = vl['y_label'].astype(np.float32)
del tr, vl; gc.collect()

print(f"Train: {X_train.shape} | ON={y_train.mean():.4f}")
print(f"Val  : {X_val.shape}   | ON={y_val.mean():.4f}")

rng         = np.random.default_rng(42)
idx_on_val  = np.where(y_val == 1)[0]
idx_off_val = np.where(y_val == 0)[0]
val_idx     = np.sort(np.concatenate([
    idx_on_val,
    rng.choice(idx_off_val, len(idx_on_val) * 10, replace=False)
]))
X_val_s = X_val[val_idx]
y_val_s = y_val[val_idx]
print(f"Val sample: {X_val_s.shape} | ON={y_val_s.mean():.4f}")

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt    = torch.exp(-bce)
        alpha = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha * (1 - pt) ** self.gamma * bce).mean()

class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).unsqueeze(1)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class KettleCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,   32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),  nn.ReLU(),
            nn.Conv1d(32,  64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        feat = self.cnn(x)
        mid  = feat[:, :, feat.size(2) // 2]
        return self.head(mid).squeeze(1)

model = KettleCNN().to(device)
print(f"\n✅ Parameters: {sum(p.numel() for p in model.parameters()):,}")

BATCH_SIZE = 512
EPOCHS     = 40
LR         = 1e-3
PATIENCE   = 6

train_loader = DataLoader(NILMDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(NILMDataset(X_val_s, y_val_s),
                          batch_size=2048, shuffle=False,
                          num_workers=2, pin_memory=True)

optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion  = FocalLoss(alpha=0.75, gamma=2.0)
scaler     = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
)

best_f1_on = 0.0
best_epoch = 0
best_path  = MODEL_DIR + 'kettle_v8_w99.pt'

print("\n" + "=" * 75)
print(f"{'Ep':>3} {'Loss':>8} {'F1_ON':>7} {'F1_mac':>7} "
      f"{'Prec':>7} {'Rec':>7} {'Thr':>5} {'LR':>9}")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            loss = criterion(model(X_b), y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            all_probs.extend(torch.sigmoid(model(X_b.to(device))).cpu().numpy())
            all_labels.extend(y_b.numpy().astype(int))

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    best_t, best_f1_ep = 0.5, 0.0
    for t in np.arange(0.1, 0.95, 0.05):
        f1 = f1_score(all_labels, (all_probs >= t).astype(int),
                      pos_label=1, average='binary', zero_division=0)
        if f1 > best_f1_ep:
            best_f1_ep, best_t = f1, t

    final_preds = (all_probs >= best_t).astype(int)
    f1_mac = f1_score(all_labels, final_preds, average='macro',  zero_division=0)
    prec   = precision_score(all_labels, final_preds,            zero_division=0)
    rec    = recall_score(all_labels, final_preds,               zero_division=0)
    is_best = best_f1_ep > best_f1_on

    print(f"{epoch:3d} {train_loss/len(train_loader):8.5f} {best_f1_ep:7.4f} "
          f"{f1_mac:7.4f} {prec:7.4f} {rec:7.4f} {best_t:5.2f} "
          f"{optimizer.param_groups[0]['lr']:9.6f}"
          + ("  ✅ BEST" if is_best else ""))

    scheduler.step(best_f1_ep)

    if is_best:
        best_f1_on = best_f1_ep
        best_epoch = epoch
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'best_f1_on': best_f1_on, 'f1_macro': f1_mac,
            'threshold': best_t, 'window_size': 99,
            'architecture': 'CNN Seq2Point w99 v8',
            'loss': 'FocalLoss(alpha=0.75, gamma=2.0)',
        }, best_path)

    if epoch - best_epoch >= PATIENCE:
        print(f"\n⏹ Early stopping @ epoch {epoch}")
        break

print("=" * 75)
print(f"\n🏆 Best sampled F1_ON = {best_f1_on:.4f} @ epoch {best_epoch}")

# ── Full Validation ───────────────────────────────────────────
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"\nLoaded epoch {ckpt['epoch']}")

full_loader = DataLoader(NILMDataset(X_val, y_val),
                         batch_size=2048, shuffle=False,
                         num_workers=2, pin_memory=True)
all_probs, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in full_loader:
        all_probs.extend(torch.sigmoid(model(X_b.to(device))).cpu().numpy())
        all_labels.extend(y_b.numpy().astype(int))

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

print("\n" + "=" * 62)
print(f"{'Thr':>6} {'F1_ON':>8} {'Prec':>8} {'Recall':>8} {'PredON':>8}")
print("=" * 62)

results = []
for t in np.arange(0.30, 0.995, 0.05):
    preds = (all_probs >= t).astype(int)
    f1    = f1_score(all_labels, preds, pos_label=1, average='binary', zero_division=0)
    prec  = precision_score(all_labels, preds, zero_division=0)
    rec   = recall_score(all_labels, preds,    zero_division=0)
    results.append((t, f1, prec, rec, preds.sum()))
    print(f"{t:6.2f} {f1:8.4f} {prec:8.4f} {rec:8.4f} {preds.sum():8,}")

best = max(results, key=lambda x: x[1])

def smooth_predictions(preds, min_dur):
    s = preds.copy()
    i = 0
    while i < len(s):
        if s[i] == 1:
            j = i
            while j < len(s) and s[j] == 1: j += 1
            if (j - i) < min_dur: s[i:j] = 0
            i = j
        else: i += 1
    return s

BEST_T = best[0]
smooth = smooth_predictions((all_probs >= BEST_T).astype(int), 1)

f1_s   = f1_score(all_labels, smooth, pos_label=1, average='binary', zero_division=0)
prec_s = precision_score(all_labels, smooth, zero_division=0)
rec_s  = recall_score(all_labels, smooth,    zero_division=0)

torch.save({
    'epoch': ckpt['epoch'], 'model_state_dict': model.state_dict(),
    'best_f1_on': f1_s, 'threshold': BEST_T, 'window_size': 99,
    'architecture': 'CNN Seq2Point w99 v8',
    'normalization': 'min-max per chunk (no global scaler)',
}, best_path)

print(f"\n{'='*45}")
print(f"  KETTLE v8 — FINAL RESULTS (window=99)")
print(f"{'='*45}")
print(f"  Architecture : CNN Seq2Point")
print(f"  Window size  : 99 minutes")
print(f"  Best Epoch   : {ckpt['epoch']}")
print(f"  Threshold    : {BEST_T:.2f}")
print(f"  F1_ON        : {f1_s:.4f}")
print(f"  Precision    : {prec_s:.4f}")
print(f"  Recall       : {rec_s:.4f}")
print(f"  True ON      : {int(all_labels.sum()):,}")
print(f"  Pred ON      : {smooth.sum():,}")
print(f"{'='*45}")

Mounted at /content/drive
✅ Device: cpu

Loading PKLs...
Train: (223245, 99) | ON=0.0909
Val  : (623919, 99)   | ON=0.0034
Val sample: (23397, 99) | ON=0.0909

✅ Parameters: 95,617

 Ep     Loss   F1_ON  F1_mac    Prec     Rec   Thr        LR


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  1  0.00937  0.9560  0.9758  0.9424  0.9699  0.70  0.001000  ✅ BEST


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  2  0.00793  0.9640  0.9802  0.9535  0.9746  0.65  0.001000  ✅ BEST


...

  7  0.00700  0.9493  0.9721  0.9356  0.9633  0.60  0.000500


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  8  0.00690  0.9564  0.9760  0.9393  0.9741  0.60  0.000500

⏹ Early stopping @ epoch 8

🏆 Best sampled F1_ON = 0.9640 @ epoch 2

Loaded epoch 2


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



   Thr    F1_ON     Prec   Recall   PredON
  0.30   0.1509   0.0816   0.9986   26,019
  0.35   0.2053   0.1144   0.9986   18,561
  0.40   0.2602   0.1496   0.9976   14,185
  0.45   0.3372   0.2029   0.9958   10,437
  0.50   0.4140   0.2615   0.9925    8,072
  0.55   0.4810   0.3178   0.9887    6,618
  0.60   0.5493   0.3811   0.9835    5,490
  0.65   0.6087   0.4426   0.9746    4,684
  0.70   0.6657   0.5093   0.9605    4,011
  0.75   0.7231   0.5898   0.9342    3,369
  0.80   0.7859   0.7016   0.8933    2,708
  0.85   0.8214   0.8512   0.7936    1,983
  0.90   0.5979   0.9634   0.4335      957
  0.95   0.0000   0.0000   0.0000        0

  KETTLE v8 — FINAL RESULTS (window=99)
  Architecture : CNN Seq2Point
  Window size  : 99 minutes
  Best Epoch   : 2
  Threshold    : 0.85
  F1_ON        : 0.8214
  Precision    : 0.8512
  Recall       : 0.7936
  True ON      : 2,127
  Pred ON      : 1,983


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, pickle, math, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

BASE      = '/content/drive/MyDrive/new obada/'
PKL_DIR   = BASE + 'data/preprocessed/'
MODEL_DIR = BASE + 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

print("\nLoading PKLs...")
with open(PKL_DIR + 'kettle_train_w99.pkl', 'rb') as f:
    tr = pickle.load(f)
with open(PKL_DIR + 'kettle_val_w99.pkl', 'rb') as f:
    vl = pickle.load(f)

X_train = tr['X'].astype(np.float32)
y_train = tr['y_label'].astype(np.float32)
X_val   = vl['X'].astype(np.float32)
y_val   = vl['y_label'].astype(np.float32)
del tr, vl; gc.collect()

print(f"Train: {X_train.shape} | ON={y_train.mean():.4f}")
print(f"Val  : {X_val.shape}   | ON={y_val.mean():.4f}")

rng         = np.random.default_rng(42)
idx_on_val  = np.where(y_val == 1)[0]
idx_off_val = np.where(y_val == 0)[0]
val_idx     = np.sort(np.concatenate([
    idx_on_val,
    rng.choice(idx_off_val, len(idx_on_val) * 10, replace=False)
]))
X_val_s = X_val[val_idx]
y_val_s = y_val[val_idx]
print(f"Val sample: {X_val_s.shape} | ON={y_val_s.mean():.4f}")

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt    = torch.exp(-bce)
        alpha = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha * (1 - pt) ** self.gamma * bce).mean()

class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).unsqueeze(1)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class KettleCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,   32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),  nn.ReLU(),
            nn.Conv1d(32,  64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        feat = self.cnn(x)
        mid  = feat[:, :, feat.size(2) // 2]
        return self.head(mid).squeeze(1)

model = KettleCNN().to(device)
print(f"\n✅ Parameters: {sum(p.numel() for p in model.parameters()):,}")

BATCH_SIZE = 512
EPOCHS     = 40
LR         = 1e-3
PATIENCE   = 6

train_loader = DataLoader(NILMDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(NILMDataset(X_val_s, y_val_s),
                          batch_size=2048, shuffle=False,
                          num_workers=2, pin_memory=True)

optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion  = FocalLoss(alpha=0.75, gamma=2.0)
scaler     = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
)

best_f1_on = 0.0
best_epoch = 0
best_path  = MODEL_DIR + 'kettle_v8_w99.pt'

print("\n" + "=" * 75)
print(f"{'Ep':>3} {'Loss':>8} {'F1_ON':>7} {'F1_mac':>7} "
      f"{'Prec':>7} {'Rec':>7} {'Thr':>5} {'LR':>9}")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            loss = criterion(model(X_b), y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            all_probs.extend(torch.sigmoid(model(X_b.to(device))).cpu().numpy())
            all_labels.extend(y_b.numpy().astype(int))

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    best_t, best_f1_ep = 0.5, 0.0
    for t in np.arange(0.1, 0.95, 0.05):
        f1 = f1_score(all_labels, (all_probs >= t).astype(int),
                      pos_label=1, average='binary', zero_division=0)
        if f1 > best_f1_ep:
            best_f1_ep, best_t = f1, t

    final_preds = (all_probs >= best_t).astype(int)
    f1_mac = f1_score(all_labels, final_preds, average='macro',  zero_division=0)
    prec   = precision_score(all_labels, final_preds,            zero_division=0)
    rec    = recall_score(all_labels, final_preds,               zero_division=0)
    is_best = best_f1_ep > best_f1_on

    print(f"{epoch:3d} {train_loss/len(train_loader):8.5f} {best_f1_ep:7.4f} "
          f"{f1_mac:7.4f} {prec:7.4f} {rec:7.4f} {best_t:5.2f} "
          f"{optimizer.param_groups[0]['lr']:9.6f}"
          + ("  ✅ BEST" if is_best else ""))

    scheduler.step(best_f1_ep)

    if is_best:
        best_f1_on = best_f1_ep
        best_epoch = epoch
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'best_f1_on': best_f1_on, 'f1_macro': f1_mac,
            'threshold': best_t, 'window_size': 99,
            'architecture': 'CNN Seq2Point w99 v8',
            'loss': 'FocalLoss(alpha=0.75, gamma=2.0)',
        }, best_path)

    if epoch - best_epoch >= PATIENCE:
        print(f"\n⏹ Early stopping @ epoch {epoch}")
        break

print("=" * 75)
print(f"\n🏆 Best sampled F1_ON = {best_f1_on:.4f} @ epoch {best_epoch}")

# ── Full Validation ───────────────────────────────────────────
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"\nLoaded epoch {ckpt['epoch']}")

full_loader = DataLoader(NILMDataset(X_val, y_val),
                         batch_size=2048, shuffle=False,
                         num_workers=2, pin_memory=True)
all_probs, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in full_loader:
        all_probs.extend(torch.sigmoid(model(X_b.to(device))).cpu().numpy())
        all_labels.extend(y_b.numpy().astype(int))

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

print("\n" + "=" * 62)
print(f"{'Thr':>6} {'F1_ON':>8} {'Prec':>8} {'Recall':>8} {'PredON':>8}")
print("=" * 62)

results = []
for t in np.arange(0.30, 0.995, 0.05):
    preds = (all_probs >= t).astype(int)
    f1    = f1_score(all_labels, preds, pos_label=1, average='binary', zero_division=0)
    prec  = precision_score(all_labels, preds, zero_division=0)
    rec   = recall_score(all_labels, preds,    zero_division=0)
    results.append((t, f1, prec, rec, preds.sum()))
    print(f"{t:6.2f} {f1:8.4f} {prec:8.4f} {rec:8.4f} {preds.sum():8,}")

best = max(results, key=lambda x: x[1])

def smooth_predictions(preds, min_dur):
    s = preds.copy()
    i = 0
    while i < len(s):
        if s[i] == 1:
            j = i
            while j < len(s) and s[j] == 1: j += 1
            if (j - i) < min_dur: s[i:j] = 0
            i = j
        else: i += 1
    return s

BEST_T = best[0]
smooth = smooth_predictions((all_probs >= BEST_T).astype(int), 1)

f1_s   = f1_score(all_labels, smooth, pos_label=1, average='binary', zero_division=0)
prec_s = precision_score(all_labels, smooth, zero_division=0)
rec_s  = recall_score(all_labels, smooth,    zero_division=0)

torch.save({
    'epoch': ckpt['epoch'], 'model_state_dict': model.state_dict(),
    'best_f1_on': f1_s, 'threshold': BEST_T, 'window_size': 99,
    'architecture': 'CNN Seq2Point w99 v8',
    'normalization': 'min-max per chunk (no global scaler)',
}, best_path)

print(f"\n{'='*45}")
print(f"  KETTLE v8 — FINAL RESULTS (window=99)")
print(f"{'='*45}")
print(f"  Architecture : CNN Seq2Point")
print(f"  Window size  : 99 minutes")
print(f"  Best Epoch   : {ckpt['epoch']}")
print(f"  Threshold    : {BEST_T:.2f}")
print(f"  F1_ON        : {f1_s:.4f}")
print(f"  Precision    : {prec_s:.4f}")
print(f"  Recall       : {rec_s:.4f}")
print(f"  True ON      : {int(all_labels.sum()):,}")
print(f"  Pred ON      : {smooth.sum():,}")
print(f"{'='*45}")

Mounted at /content/drive
✅ Device: cuda

Loading PKLs...
Train: (223245, 99) | ON=0.0909
Val  : (623919, 99)   | ON=0.0034
Val sample: (23397, 99) | ON=0.0909

✅ Parameters: 95,617

 Ep     Loss   F1_ON  F1_mac    Prec     Rec   Thr        LR
  1  0.00991  0.9335  0.9634  0.9232  0.9441  0.65  0.001000  ✅ BEST
  2  0.00786  0.9641  0.9802  0.9561  0.9723  0.70  0.001000  ✅ BEST
  3  0.00762  0.9482  0.9714  0.9295  0.9676  0.60  0.001000
  4  0.00756  0.9502  0.9726  0.9279  0.9737  0.55  0.001000
  5  0.00739  0.9619  0.9790  0.9563  0.9676  0.65  0.001000
  6  0.00730  0.9570  0.9763  0.9413  0.9732  0.65  0.001000
  7  0.00690  0.9441  0.9692  0.9209  0.9685  0.55  0.000500
  8  0.00686  0.9512  0.9732  0.9440  0.9586  0.65  0.000500

⏹ Early stopping @ epoch 8

🏆 Best sampled F1_ON = 0.9641 @ epoch 2

Loaded epoch 2

   Thr    F1_ON     Prec   Recall   PredON
  0.30   0.1596   0.0867   0.9986   24,491
  0.35   0.2074   0.1157   0.9981   18,347
  0.40   0.2548   0.1460   0.9976   1

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, pickle, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

BASE      = '/content/drive/MyDrive/new obada/'
PKL_DIR   = BASE + 'data/preprocessed/'
MODEL_DIR = BASE + 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

print("\nLoading PKLs...")
with open(PKL_DIR + 'kettle_train_w49.pkl', 'rb') as f:
    tr = pickle.load(f)
with open(PKL_DIR + 'kettle_val_w49.pkl', 'rb') as f:
    vl = pickle.load(f)

X_train = tr['X'].astype(np.float32)
y_train = tr['y_label'].astype(np.float32)
X_val   = vl['X'].astype(np.float32)
y_val   = vl['y_label'].astype(np.float32)
del tr, vl; gc.collect()

# [AR_COMMENT_REMOVED]
assert X_train.shape[1] == 49, "shape mismatch"
assert X_val.shape[1]   == 49, "shape mismatch"

print(f"Train: {X_train.shape} | ON={y_train.mean():.4f}")
print(f"Val  : {X_val.shape}   | ON={y_val.mean():.4f}")

rng         = np.random.default_rng(42)
idx_on_val  = np.where(y_val == 1)[0]
idx_off_val = np.where(y_val == 0)[0]
val_idx     = np.sort(np.concatenate([
    idx_on_val,
    rng.choice(idx_off_val, len(idx_on_val) * 10, replace=False)
]))
X_val_s = X_val[val_idx]
y_val_s = y_val[val_idx]
print(f"Val sample: {X_val_s.shape} | ON={y_val_s.mean():.4f}")

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt    = torch.exp(-bce)
        alpha = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha * (1 - pt) ** self.gamma * bce).mean()

class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).unsqueeze(1)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class KettleCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,   32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),  nn.ReLU(),
            nn.Conv1d(32,  64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        feat = self.cnn(x)
        mid  = feat[:, :, feat.size(2) // 2]
        return self.head(mid).squeeze(1)

model = KettleCNN().to(device)
print(f"\n✅ Parameters: {sum(p.numel() for p in model.parameters()):,}")

BATCH_SIZE = 512
EPOCHS     = 40
LR         = 1e-3
PATIENCE   = 6

train_loader = DataLoader(NILMDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(NILMDataset(X_val_s, y_val_s),
                          batch_size=2048, shuffle=False,
                          num_workers=2, pin_memory=True)

optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion  = FocalLoss(alpha=0.75, gamma=2.0)
scaler     = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
)

best_f1_on = 0.0
best_epoch = 0
best_path  = MODEL_DIR + 'kettle_v9_w49.pt'

print("\n" + "=" * 75)
print(f"{'Ep':>3} {'Loss':>8} {'F1_ON':>7} {'F1_mac':>7} "
      f"{'Prec':>7} {'Rec':>7} {'Thr':>5} {'LR':>9}")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            loss = criterion(model(X_b), y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            all_probs.extend(torch.sigmoid(model(X_b.to(device))).cpu().numpy())
            all_labels.extend(y_b.numpy().astype(int))

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    best_t, best_f1_ep = 0.5, 0.0
    for t in np.arange(0.1, 0.95, 0.05):
        f1 = f1_score(all_labels, (all_probs >= t).astype(int),
                      pos_label=1, average='binary', zero_division=0)
        if f1 > best_f1_ep:
            best_f1_ep, best_t = f1, t

    final_preds = (all_probs >= best_t).astype(int)
    f1_mac = f1_score(all_labels, final_preds, average='macro',  zero_division=0)
    prec   = precision_score(all_labels, final_preds,            zero_division=0)
    rec    = recall_score(all_labels, final_preds,               zero_division=0)
    is_best = best_f1_ep > best_f1_on

    print(f"{epoch:3d} {train_loss/len(train_loader):8.5f} {best_f1_ep:7.4f} "
          f"{f1_mac:7.4f} {prec:7.4f} {rec:7.4f} {best_t:5.2f} "
          f"{optimizer.param_groups[0]['lr']:9.6f}"
          + ("  ✅ BEST" if is_best else ""))

    scheduler.step(best_f1_ep)

    if is_best:
        best_f1_on = best_f1_ep
        best_epoch = epoch
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'best_f1_on': best_f1_on, 'f1_macro': f1_mac,
            'threshold': best_t, 'window_size': 49,
            'architecture': 'CNN Seq2Point w49 v9',
            'loss': 'FocalLoss(alpha=0.75, gamma=2.0)',
        }, best_path)

    if epoch - best_epoch >= PATIENCE:
        print(f"\n⏹ Early stopping @ epoch {epoch}")
        break

print("=" * 75)
print(f"\n🏆 Best sampled F1_ON = {best_f1_on:.4f} @ epoch {best_epoch}")

# ── Full Validation ───────────────────────────────────────────
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"\nLoaded epoch {ckpt['epoch']}")

full_loader = DataLoader(NILMDataset(X_val, y_val),
                         batch_size=2048, shuffle=False,
                         num_workers=2, pin_memory=True)
all_probs, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in full_loader:
        all_probs.extend(torch.sigmoid(model(X_b.to(device))).cpu().numpy())
        all_labels.extend(y_b.numpy().astype(int))

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

print("\n" + "=" * 62)
print(f"{'Thr':>6} {'F1_ON':>8} {'Prec':>8} {'Recall':>8} {'PredON':>8}")
print("=" * 62)

results = []
for t in np.arange(0.30, 0.995, 0.05):
    preds = (all_probs >= t).astype(int)
    f1    = f1_score(all_labels, preds, pos_label=1, average='binary', zero_division=0)
    prec  = precision_score(all_labels, preds, zero_division=0)
    rec   = recall_score(all_labels, preds,    zero_division=0)
    results.append((t, f1, prec, rec, preds.sum()))
    print(f"{t:6.2f} {f1:8.4f} {prec:8.4f} {rec:8.4f} {preds.sum():8,}")

best = max(results, key=lambda x: x[1])

def smooth_predictions(preds, min_dur):
    s = preds.copy()
    i = 0
    while i < len(s):
        if s[i] == 1:
            j = i
            while j < len(s) and s[j] == 1: j += 1
            if (j - i) < min_dur: s[i:j] = 0
            i = j
        else: i += 1
    return s

BEST_T = best[0]
smooth = smooth_predictions((all_probs >= BEST_T).astype(int), 1)
f1_s   = f1_score(all_labels, smooth, pos_label=1, average='binary', zero_division=0)
prec_s = precision_score(all_labels, smooth, zero_division=0)
rec_s  = recall_score(all_labels, smooth,    zero_division=0)

torch.save({
    'epoch': ckpt['epoch'], 'model_state_dict': model.state_dict(),
    'best_f1_on': f1_s, 'threshold': BEST_T, 'window_size': 49,
    'architecture': 'CNN Seq2Point w49 v9',
    'normalization': 'min-max per chunk (no global scaler)',
}, best_path)

print(f"\n{'='*45}")
print(f"  KETTLE v9 — FINAL RESULTS (window=49)")
print(f"{'='*45}")
print(f"  Architecture : CNN Seq2Point")
print(f"  Window size  : 49 minutes")
print(f"  Best Epoch   : {ckpt['epoch']}")
print(f"  Threshold    : {BEST_T:.2f}")
print(f"  F1_ON        : {f1_s:.4f}")
print(f"  Precision    : {prec_s:.4f}")
print(f"  Recall       : {rec_s:.4f}")
print(f"  True ON      : {int(all_labels.sum()):,}")
print(f"  Pred ON      : {smooth.sum():,}")
print(f"{'='*45}")

Mounted at /content/drive
✅ Device: cuda

Loading PKLs...
Train: (223256, 49) | ON=0.0909
Val  : (623969, 49)   | ON=0.0034
Val sample: (23397, 49) | ON=0.0909

✅ Parameters: 91,457

 Ep     Loss   F1_ON  F1_mac    Prec     Rec   Thr        LR
  1  0.00935  0.9516  0.9733  0.9424  0.9610  0.65  0.001000  ✅ BEST
  2  0.00792  0.9651  0.9808  0.9621  0.9680  0.70  0.001000  ✅ BEST
  3  0.00771  0.9528  0.9740  0.9471  0.9586  0.70  0.001000
  4  0.00755  0.9548  0.9751  0.9323  0.9784  0.55  0.001000
  5  0.00743  0.9582  0.9770  0.9363  0.9812  0.55  0.001000
  6  0.00733  0.9642  0.9803  0.9536  0.9751  0.65  0.001000
  7  0.00701  0.9472  0.9709  0.9298  0.9652  0.60  0.000500
  8  0.00699  0.9541  0.9747  0.9406  0.9680  0.60  0.000500

⏹ Early stopping @ epoch 8

🏆 Best sampled F1_ON = 0.9651 @ epoch 2

Loaded epoch 2

   Thr    F1_ON     Prec   Recall   PredON
  0.30   0.1617   0.0880   0.9981   24,131
  0.35   0.2122   0.1187   0.9981   17,883
  0.40   0.2666   0.1539   0.9972   1

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

BASE      = '/content/drive/MyDrive/new obada/'
PKL_DIR   = BASE + 'data/preprocessed/'
MODEL_DIR = BASE + 'models/'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load val ─────────────────────────────────────────────────
with open(PKL_DIR + 'kettle_val_w99.pkl', 'rb') as f:
    vl = pickle.load(f)
X_val   = vl['X'].astype(np.float32)
y_val_l = vl['y_label'].astype(np.float32)
y_val_w = vl['y_watts'].astype(np.float32)

# [AR_COMMENT_REMOVED]
with open(PKL_DIR + 'kettle_train_w99.pkl', 'rb') as f:
    tr = pickle.load(f)
y_train_w = tr['y_watts'].astype(np.float32)
y_train_l = tr['y_label'].astype(np.float32)

on_watts = y_train_w[y_train_l == 1]
KETTLE_AVG_WATTS = float(on_watts.mean())
KETTLE_STD_WATTS = float(on_watts.std())

print(f"Kettle avg ON watts : {KETTLE_AVG_WATTS:.0f} W")
print(f"Kettle std ON watts : {KETTLE_STD_WATTS:.0f} W")
print(f"Kettle min ON watts : {on_watts.min():.0f} W")
print(f"Kettle max ON watts : {on_watts.max():.0f} W")

# ── Model ─────────────────────────────────────────────────────
class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).unsqueeze(1)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class KettleCNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,   32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),  nn.ReLU(),
            nn.Conv1d(32,  64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        feat = self.cnn(x)
        mid  = feat[:, :, feat.size(2) // 2]
        return self.head(mid).squeeze(1)

ckpt  = torch.load(MODEL_DIR + 'kettle_v8_w99.pt',
                   map_location=device, weights_only=False)
model = KettleCNN().to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
THRESHOLD = ckpt['threshold']
print(f"\nLoaded: {ckpt['architecture']} | threshold={THRESHOLD}")

# ── Inference ─────────────────────────────────────────────────
full_loader = DataLoader(NILMDataset(X_val, y_val_l),
                         batch_size=2048, shuffle=False,
                         num_workers=2, pin_memory=True)
all_probs, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in full_loader:
        all_probs.extend(torch.sigmoid(model(X_b.to(device))).cpu().numpy())
        all_labels.extend(y_b.numpy().astype(int))

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
pred_labels = (all_probs >= THRESHOLD).astype(int)

# ── F1 metrics ────────────────────────────────────────────────
f1   = f1_score(all_labels, pred_labels, pos_label=1, average='binary', zero_division=0)
prec = precision_score(all_labels, pred_labels, zero_division=0)
rec  = recall_score(all_labels, pred_labels,    zero_division=0)

# ── Power Estimation ──────────────────────────────────────────
# [AR_COMMENT_REMOVED]
pred_watts = pred_labels * KETTLE_AVG_WATTS
true_watts = y_val_w

mae         = np.mean(np.abs(pred_watts - true_watts))
true_kwh    = true_watts.sum() / 60 / 1000
pred_kwh    = pred_watts.sum() / 60 / 1000
kwh_error   = abs(true_kwh - pred_kwh)
kwh_error_pct = kwh_error / max(true_kwh, 0.01) * 100

# [AR_COMMENT_REMOVED]
def count_events(labels):
    events = []
    i = 0
    while i < len(labels):
        if labels[i] == 1:
            start = i
            while i < len(labels) and labels[i] == 1:
                i += 1
            events.append((start, i - 1, i - start))
        else:
            i += 1
    return events

true_events = count_events(all_labels)
pred_events = count_events(pred_labels)

# ── Results ───────────────────────────────────────────────────
print("\n" + "=" * 50)
print("  KETTLE — FINAL EVALUATION")
print("=" * 50)
print(f"\n── Detection ──")
print(f"  F1_ON        : {f1:.4f}")
print(f"  Precision    : {prec:.4f}")
print(f"  Recall       : {rec:.4f}")
print(f"  Threshold    : {THRESHOLD:.2f}")

print(f"\n── Events ──")
print(f"  True events  : {len(true_events):,}")
print(f"  Pred events  : {len(pred_events):,}")
avg_true_dur = np.mean([e[2] for e in true_events]) if true_events else 0
avg_pred_dur = np.mean([e[2] for e in pred_events]) if pred_events else 0
print(f"  Avg true dur : {avg_true_dur:.1f} min")
print(f"  Avg pred dur : {avg_pred_dur:.1f} min")

print(f"\n── Power & Energy ──")
print(f"  Kettle avg   : {KETTLE_AVG_WATTS:.0f} W")
print(f"  MAE          : {mae:.1f} W")
print(f"  True ON min  : {int(all_labels.sum()):,} minutes")
print(f"  Pred ON min  : {int(pred_labels.sum()):,} minutes")
print(f"  True kWh     : {true_kwh:.3f} kWh")
print(f"  Pred kWh     : {pred_kwh:.3f} kWh")
print(f"  kWh error    : {kwh_error:.3f} kWh ({kwh_error_pct:.1f}%)")
print("=" * 50)

# [AR_COMMENT_REMOVED]
ckpt['avg_on_watts']  = KETTLE_AVG_WATTS
ckpt['std_on_watts']  = KETTLE_STD_WATTS
ckpt['final_f1']      = f1
ckpt['final_mae']     = mae
ckpt['true_kwh_val']  = true_kwh
ckpt['pred_kwh_val']  = pred_kwh
ckpt['kwh_error_pct'] = kwh_error_pct

torch.save(ckpt, MODEL_DIR + 'kettle_v8_w99.pt')
print(f"\n✅ Updated model saved with power metadata")

Mounted at /content/drive
Kettle avg ON watts : 2478 W
Kettle std ON watts : 298 W
Kettle min ON watts : 2000 W
Kettle max ON watts : 3026 W

Loaded: CNN Seq2Point w99 v8 | threshold=0.8499999999999999

  KETTLE — FINAL EVALUATION

── Detection ──
  F1_ON        : 0.7844
  Precision    : 0.7262
  Recall       : 0.8528
  Threshold    : 0.85

── Events ──
  True events  : 2,122
  Pred events  : 2,491
  Avg true dur : 1.0 min
  Avg pred dur : 1.0 min

── Power & Energy ──
  Kettle avg   : 2478 W
  MAE          : 7.2 W
  True ON min  : 2,127 minutes
  Pred ON min  : 2,498 minutes
  True kWh     : 135.467 kWh
  Pred kWh     : 103.173 kWh
  kWh error    : 32.295 kWh (23.8%)

✅ Updated model saved with power metadata


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import torch

MODEL_DIR = '/content/drive/MyDrive/new obada/models/'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt = torch.load(MODEL_DIR + 'kettle_8sec_seq2point.pt',
                  map_location=device, weights_only=False)

# [AR_COMMENT_REMOVED]
ckpt['final_full_val_f1']   = 0.9125
ckpt['final_precision']     = 0.8952
ckpt['final_recall']        = 0.9306
ckpt['final_threshold']     = 0.75
ckpt['avg_on_watts']        = 2478.0
ckpt['train_houses']        = [2,3,5,6,7,8,9,11,13,17,19,21]
ckpt['val_house']           = 20
ckpt['data_resolution']     = '8-second'
ckpt['window_size']         = 599
ckpt['normalization']       = 'min-max per house'
ckpt['imbalance']           = '1:10'
ckpt['loss']                = 'FocalLoss(alpha=0.75, gamma=2.0)'

torch.save(ckpt, MODEL_DIR + 'kettle_8sec_seq2point.pt')

print("="*50)
print("  KETTLE — FINAL RESULTS")
print("="*50)
print(f"  Architecture : Zhang et al. CNN Seq2Point")
print(f"  Data         : 8-second raw REFIT")
print(f"  Train houses : 12 houses")
print(f"  Val house    : House 20 (unseen)")
print(f"  Window size  : 599 samples (~80 min)")
print(f"  Threshold    : 0.75")
print(f"  F1_ON        : 0.9125")
print(f"  Precision    : 0.8952")
print(f"  Recall       : 0.9306")
print(f"  True ON      : 24,013")
print(f"  Pred ON      : 24,963")
print(f"  Model file   : kettle_8sec_seq2point.pt")
print("="*50)
print("\n✅ Saved!")

Mounted at /content/drive
  KETTLE — FINAL RESULTS
  Architecture : Zhang et al. CNN Seq2Point
  Data         : 8-second raw REFIT
  Train houses : 12 houses
  Val house    : House 20 (unseen)
  Window size  : 599 samples (~80 min)
  Threshold    : 0.75
  F1_ON        : 0.9125
  Precision    : 0.8952
  Recall       : 0.9306
  True ON      : 24,013
  Pred ON      : 24,963
  Model file   : kettle_8sec_seq2point.pt

✅ Saved!
